# Backfill snapshot streaming lama ke Supabase ETL (sekali jalan)

GitHub Actions baru mengambil GBFS sejak pipeline aktif. Snapshot yang sudah dikumpulkan sebelumnya (polling notebook dan Apps Script, 28 Sep 2026 sampai sekarang) ada di Google Drive, jadi dimuat sekali lewat notebook ini **memakai transform dan load yang sama** dengan pipeline (`etl/transform.py`, `etl/load.py`).

Password database diketik sendiri saat diminta (tidak disimpan di notebook).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf citibike-jc-pipeline && git clone -q https://github.com/rafiptrnto/citibike-jc-pipeline.git
%cd citibike-jc-pipeline
!pip -q install -r requirements.txt

In [ ]:
import os, getpass
os.environ['DATABASE_URL'] = getpass.getpass('Connection string Supabase (Session pooler): ')

In [ ]:
CSV = '/content/drive/MyDrive/MATPIL BIG DATA/dataset_final/dataset_final_streaming.csv'
!python -m etl.run_etl backfill-stream "$CSV"

In [ ]:
import pandas as pd
from etl import load
c = load.connect()
display(load.query_df(c, "select sumber, count(*) baris, min(waktu), max(waktu) from public.fact_station_30min group by 1"))
display(load.query_df(c, "select job, mulai, status, baris_extract, baris_transform, baris_load, pesan from public.etl_log_run order by id desc limit 5"))
c.close()